# IT Service Desk & SLA Analytics

**Portfolio case study | Python, Pandas, NumPy, Matplotlib, EDA and operational KPI reporting**

This notebook analyses a synthetic service desk dataset to understand ticket demand, response and resolution performance, backlog, agent workload, and resolution SLA attainment.

> **Data note:** The dataset is synthetic and illustrative. It does not represent a real organisation or production service desk.

**Expected file:** `data/service_desk_tickets.csv`

## 1. Business objective

Assess service desk performance using operational KPIs and identify where further investigation or process improvement may be useful.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the dataset

The notebook checks common repository locations so it can run from either the repository root or the `notebooks/` directory.

In [ ]:
candidate_paths = [
    Path("data/service_desk_tickets.csv"),
    Path("../data/service_desk_tickets.csv"),
    Path("service_desk_tickets.csv")
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find data/service_desk_tickets.csv. "
        "Place the CSV in the repository's data/ folder."
    )

df = pd.read_csv(data_path, parse_dates=["created_at"])
print(f"Loaded: {data_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())


## 3. Understand the data

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("Column names:", df.columns.tolist())


## 4. Data quality checks

Check missing values, duplicate ticket identifiers, and basic business-rule consistency before calculating KPIs.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "data_type": df.dtypes.astype(str)
})
display(quality_summary)

print("Duplicate ticket IDs:", df["ticket_id"].duplicated().sum())
print("Negative response times:", (df["response_time_min"] < 0).sum())
print("Negative resolution times:", (df["resolution_time_min"] < 0).sum())
print("Invalid SLA flags:", (~df["resolution_sla_met"].isin([0, 1])).sum())
print("Rows with SLA target <= 0:", (df["resolution_sla_min"] <= 0).sum())


In [ ]:
# Work on a copy and standardise text fields used in the analysis.
df["category"] = df["category"].astype("string").str.strip()
df["priority"] = df["priority"].astype("string").str.strip().str.upper()
df["status"] = df["status"].astype("string").str.strip().str.title()
df["assigned_to"] = df["assigned_to"].astype("string").str.strip()

# Convert numeric measures safely.
numeric_cols = [
    "response_time_min", "resolution_time_min",
    "resolution_sla_min", "resolution_sla_met"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Derive useful time dimensions.
df["month"] = df["created_at"].dt.to_period("M").astype(str)
df["year"] = df["created_at"].dt.year

completed_statuses = ["Resolved", "Closed"]
open_statuses = ["In Progress", "Pending"]
completed = df[df["status"].isin(completed_statuses)].copy()
open_tickets = df[df["status"].isin(open_statuses)].copy()

print(f"Completed tickets: {len(completed):,}")
print(f"Open backlog tickets: {len(open_tickets):,}")


## 5. Define and calculate core KPIs

SLA attainment is calculated only for completed tickets, using the provided `resolution_sla_met` indicator.

In [ ]:
total_tickets = df["ticket_id"].nunique()
completed_count = len(completed)
open_backlog = len(open_tickets)
unassigned_count = df["assigned_to"].str.lower().eq("unassigned").sum()

avg_response = df["response_time_min"].mean()
avg_resolution_completed = completed["resolution_time_min"].mean()

sla_attainment = (
    completed["resolution_sla_met"].mean() * 100
    if completed_count and completed["resolution_sla_met"].notna().any()
    else np.nan
)

kpi_summary = pd.DataFrame({
    "KPI": [
        "Unique tickets created",
        "Completed tickets",
        "Open backlog",
        "Unassigned tickets",
        "Average response time (minutes)",
        "Average resolution time for completed tickets (minutes)",
        "Resolution SLA attainment (%)"
    ],
    "Value": [
        total_tickets, completed_count, open_backlog, unassigned_count,
        avg_response, avg_resolution_completed, sla_attainment
    ]
})
display(kpi_summary)


## 6. Ticket demand analysis

In [ ]:
category_volume = df["category"].value_counts().sort_values(ascending=False)
priority_volume = df["priority"].value_counts().sort_index()
status_volume = df["status"].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
category_volume.plot(kind="bar", ax=axes[0], title="Tickets by category")
priority_volume.plot(kind="bar", ax=axes[1], title="Tickets by priority")
status_volume.plot(kind="bar", ax=axes[2], title="Tickets by status")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("Ticket count")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

display(category_volume.rename("ticket_count").to_frame())


## 7. Monthly ticket trend

In [ ]:
monthly_volume = df.groupby("month")["ticket_id"].nunique().sort_index()
plt.figure(figsize=(12, 4))
monthly_volume.plot(marker="o")
plt.title("Monthly ticket volume")
plt.xlabel("Month")
plt.ylabel("Unique tickets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_volume.rename("unique_tickets").to_frame())


## 8. Response and resolution performance by priority

In [ ]:
priority_performance = df.groupby("priority").agg(
    ticket_count=("ticket_id", "nunique"),
    avg_response_min=("response_time_min", "mean"),
    avg_resolution_min=("resolution_time_min", "mean")
).round(2)
display(priority_performance)

priority_performance[["avg_response_min", "avg_resolution_min"]].plot(
    kind="bar", figsize=(9, 4), title="Average response and resolution time by priority"
)
plt.xlabel("Priority")
plt.ylabel("Minutes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. SLA attainment by category and priority

In [ ]:
sla_by_category = completed.groupby("category").agg(
    completed_tickets=("ticket_id", "nunique"),
    sla_attainment_pct=("resolution_sla_met", lambda s: s.mean() * 100)
).round(2).sort_values("sla_attainment_pct")

sla_by_priority = completed.groupby("priority").agg(
    completed_tickets=("ticket_id", "nunique"),
    sla_attainment_pct=("resolution_sla_met", lambda s: s.mean() * 100)
).round(2).sort_index()

display(sla_by_category)
display(sla_by_priority)

sla_by_category["sla_attainment_pct"].plot(
    kind="bar", figsize=(9, 4), title="Resolution SLA attainment by category"
)
plt.xlabel("Category")
plt.ylabel("SLA attainment (%)")
plt.ylim(0, 100)
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


## 10. Agent workload and unassigned queue

In [ ]:
agent_workload = df.groupby("assigned_to").agg(
    total_tickets=("ticket_id", "nunique"),
    completed_tickets=("status", lambda s: s.isin(completed_statuses).sum()),
    open_tickets=("status", lambda s: s.isin(open_statuses).sum()),
    avg_resolution_min=("resolution_time_min", "mean")
).round(2).sort_values("total_tickets", ascending=False)

display(agent_workload)

agent_workload["total_tickets"].plot(
    kind="bar", figsize=(10, 4), title="Ticket workload by assignee"
)
plt.xlabel("Assignee")
plt.ylabel("Ticket count")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


## 11. Build an investigation summary

The following tables identify areas to investigate. They do not, by themselves, establish the cause of delays or SLA misses.

In [ ]:
print("Highest-volume category:")
display(category_volume.head(1))

print("Priority with the longest average resolution time:")
display(priority_performance["avg_resolution_min"].sort_values(ascending=False).head(1))

print("Categories with the lowest SLA attainment (minimum 1 completed ticket):")
display(sla_by_category.sort_values("sla_attainment_pct").head(3))

print("Unassigned tickets:", int(unassigned_count))


## 12. Business interpretation and recommendations

Use the actual outputs above to complete this section. Avoid assuming a category, priority, or agent is underperforming without checking the calculated results.

Suggested questions for a service desk lead:
- Which incident categories account for the largest share of demand?
- Do high-priority tickets receive faster responses and resolutions?
- Which categories or priorities have lower resolution SLA attainment?
- How many tickets are pending or in progress, and are any unassigned?
- Are monthly changes in ticket volume associated with changes in response or resolution time?

Potential actions to validate with stakeholders:
- Review routing and assignment rules for unassigned tickets.
- Examine recurring incident categories and assess whether knowledge articles or self-service could reduce demand.
- Review staffing and escalation coverage for priority groups with longer resolution times.
- Monitor SLA attainment alongside ticket volume and backlog rather than using a single KPI in isolation.

In [ ]:
# Optional: export analysis tables for reporting.
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

kpi_summary.to_csv(output_dir / "kpi_summary.csv", index=False)
priority_performance.to_csv(output_dir / "priority_performance.csv")
sla_by_category.to_csv(output_dir / "sla_by_category.csv")
agent_workload.to_csv(output_dir / "agent_workload.csv")

print(f"Analysis tables saved to: {output_dir.resolve()}")


## 13. Conclusion

This notebook demonstrates a repeatable approach to service desk data analysis: data quality checks, KPI calculation, trend analysis, SLA monitoring, workload review, and operational recommendations.

**Portfolio disclosure:** This is an independent learning project using synthetic data. Numerical conclusions should be taken from the executed notebook output, not assumed in advance.